# 0504 · 00 · Preparar el entorno

Laboratorio 0504 · Ingesta de CSV y la columna de datos rescatados.

Ejecuta este notebook con **Run all** antes de empezar. Hace cuatro cosas:

1. Comprueba que en la carpeta `datasets/` del repositorio están:
   - los cuatro CSV de ventas de `sales-csv/`, con sus filas esperadas;
   - los dos CSV con defectos de `csv-demo-files/`, con el defecto que necesita la demo.
2. Crea el catálogo `lab0504` con dos schemas:
   - `bronze`: donde se crean las tablas del laboratorio.
   - `upload`: con el volumen `upload`, la carpeta de aterrizaje de los ficheros.
3. Deja el laboratorio en su **estado inicial**: borra las tablas de una ejecución anterior, vacía el volumen y copia las dos carpetas.
4. Verifica que el volumen contiene **12.484 filas** de ventas.

Si quieres repetir el laboratorio desde cero, vuelve a ejecutarlo: siempre deja el mismo punto de partida.

In [ ]:
import os

CATALOGO = "lab0504"
VOLUMEN_UPLOAD = f"/Volumes/{CATALOGO}/upload/upload"

# La carpeta de este notebook es la del laboratorio dentro de tu Git folder;
# los datasets viven un nivel más arriba, en la raíz del repositorio.
RUTA_LAB = os.getcwd()
if not RUTA_LAB.startswith("/Workspace"):
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    RUTA_LAB = "/Workspace" + os.path.dirname(ctx.notebookPath().get())
RUTA_DATASETS = f"{os.path.dirname(RUTA_LAB)}/datasets"

print("Laboratorio:", RUTA_LAB)
print("Datasets:   ", RUTA_DATASETS)
print("Catálogo:   ", CATALOGO)

## 1 · Comprobar los ficheros del repositorio

Los CSV de ventas se cuentan con un lector CSV real (no por líneas). De los dos ficheros con defectos se comprueba justo el defecto que usa la demo: si alguien los abre y los guarda con otra herramienta, el defecto podría desaparecer.

In [ ]:
import csv

FILAS_VENTAS = {"000.csv": 3149, "001.csv": 2932, "002.csv": 3200, "003.csv": 3203}

if not os.path.isdir(RUTA_DATASETS):
    raise FileNotFoundError(
        f"No encuentro {RUTA_DATASETS}. Comprueba que tu Git folder está actualizado "
        "y que la carpeta datasets/ está en la raíz del repositorio.")

errores = []
for fichero, esperadas in FILAS_VENTAS.items():
    ruta = f"{RUTA_DATASETS}/sales-csv/{fichero}"
    if not os.path.exists(ruta):
        errores.append(f"falta sales-csv/{fichero}")
        continue
    with open(ruta, newline="", encoding="utf-8") as f:
        lector = csv.reader(f, delimiter="|", quotechar='"', escapechar="\\", doublequote=False)
        filas = sum(1 for _ in lector) - 1  # sin la cabecera
    print(f"sales-csv/{fichero}: {filas} filas")
    if filas != esperadas:
        errores.append(f"sales-csv/{fichero} tiene {filas} filas, esperaba {esperadas}")

# Defectos controlados: cuatro filas de datos y la primera línea esperada
DEFECTOS = {
    "malformed_example_1_data.csv": ("order_id|email|transactions_timestamp", "|aaa"),
    "malformed_example_2_data.csv": ("|email|transactions_timestamp", None),
}
for fichero, (cabecera, fin_fila_1) in DEFECTOS.items():
    ruta = f"{RUTA_DATASETS}/csv-demo-files/{fichero}"
    if not os.path.exists(ruta):
        errores.append(f"falta csv-demo-files/{fichero}")
        continue
    with open(ruta, encoding="utf-8") as f:
        lineas = f.read().splitlines()
    ok = (len(lineas) == 5 and lineas[0] == cabecera
          and (fin_fila_1 is None or lineas[1].endswith(fin_fila_1)))
    print(f"csv-demo-files/{fichero}: cabecera {lineas[0]!r}  {'ok' if ok else 'DEFECTO ALTERADO'}")
    if not ok:
        errores.append(f"csv-demo-files/{fichero} no tiene el defecto esperado")

if errores:
    raise ValueError("Ficheros incorrectos: " + "; ".join(errores))

## 2 · Catálogo, schemas y volumen

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO}")
for schema in ["bronze", "upload"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.upload.upload")

display(spark.sql(f"SHOW SCHEMAS IN {CATALOGO}"))

## 3 · Volver al estado inicial

In [ ]:
for tabla in ["sales_bronze", "demo_4_example_2_bronze"]:
    spark.sql(f"DROP TABLE IF EXISTS {CATALOGO}.bronze.{tabla}")

for item in dbutils.fs.ls(VOLUMEN_UPLOAD):
    dbutils.fs.rm(item.path, recurse=True)
print("vaciado", VOLUMEN_UPLOAD)

## 4 · Cargar los ficheros y verificar

In [ ]:
import shutil

for carpeta, ficheros in [("sales-csv", FILAS_VENTAS), ("csv-demo-files", DEFECTOS)]:
    os.makedirs(f"{VOLUMEN_UPLOAD}/{carpeta}", exist_ok=True)
    for fichero in ficheros:
        shutil.copyfile(f"{RUTA_DATASETS}/{carpeta}/{fichero}", f"{VOLUMEN_UPLOAD}/{carpeta}/{fichero}")
        print("copiado", f"{carpeta}/{fichero}")

filas = spark.sql(f"""
    SELECT count(*) AS filas
    FROM read_files('{VOLUMEN_UPLOAD}/sales-csv', format => 'csv', sep => '|', header => true)
""").first()["filas"]

assert filas == sum(FILAS_VENTAS.values()), f"El volumen tiene {filas} filas de ventas, esperaba 12484"
print(f"Entorno listo: {VOLUMEN_UPLOAD} contiene sales-csv/ ({filas} filas) y csv-demo-files/ (2 ficheros).")